# Next-Hour Bike Count Model Evaluation

This notebook loads the submitted model for the next-hour prediction task and evaluates it on a test `.xlsx` file with ground-truth `BikeCount` values.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error

TEST_PATH = None  # Set this to the provided hidden test file, e.g. "hidden_test.xlsx"
MODEL_PATH = "model_group18.joblib"

HORIZON = 1
WEATHER_CATS = ['Thunder', 'Snow', 'Rain', 'Fog', 'Clear', 'Cloudy', 'Other']
REQUIRED_COLS = ['Month', 'Day', 'Hour', 'Weekday', 'Weather', 'Temperature (°C)', 'Humidity (%)', 'Rain (mm)', 'Wind (km/h)', 'BikeCount']


In [ ]:
def load_clean(path):
    df = pd.read_excel(path)
    df.columns = df.columns.str.strip()

    missing = [col for col in REQUIRED_COLS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns {missing}. Got {list(df.columns)}")

    df = df.dropna(subset=["BikeCount"]).copy()
    df = df.sort_values(["Month", "Day", "Hour"]).reset_index(drop=True)

    duplicates = df.duplicated(["Month", "Day", "Hour"]).sum()
    if duplicates:
        raise ValueError(f"Found {duplicates} duplicated timestamps after cleaning.")

    return df


def weather_bucket(value):
    text = str(value).lower()
    if "thunder" in text:
        return "Thunder"
    if "snow" in text or "ice" in text or "sleet" in text:
        return "Snow"
    if "rain" in text or "drizzle" in text or "shower" in text:
        return "Rain"
    if "fog" in text:
        return "Fog"
    if "sunny" in text or "clear" in text:
        return "Clear"
    if "cloud" in text or "overcast" in text:
        return "Cloudy"
    return "Other"


In [ ]:
def make_supervised_frame(df):
    d = df.copy()
    horizon = HORIZON

    d["target"] = d["BikeCount"].shift(-horizon)
    d["target_hour"] = d["Hour"].shift(-horizon)
    d["target_weekday"] = d["Weekday"].shift(-horizon)
    d["target_month"] = d["Month"].shift(-horizon)
    d["target_temperature"] = d["Temperature (°C)"].shift(-horizon)
    d["target_humidity"] = d["Humidity (%)"].shift(-horizon)
    d["target_rain"] = d["Rain (mm)"].shift(-horizon)
    d["target_wind"] = d["Wind (km/h)"].shift(-horizon)
    d["target_weather"] = d["Weather"].shift(-horizon)

    d["target_is_weekend"] = d["target_weekday"].isin([5, 6]).astype(int)
    d["target_hour_sin"] = np.sin(2 * np.pi * d["target_hour"] / 24)
    d["target_hour_cos"] = np.cos(2 * np.pi * d["target_hour"] / 24)
    d["target_month_sin"] = np.sin(2 * np.pi * d["target_month"] / 12)
    d["target_month_cos"] = np.cos(2 * np.pi * d["target_month"] / 12)

    weather = pd.Categorical(d["target_weather"].map(weather_bucket), categories=WEATHER_CATS)
    weather_dummies = pd.get_dummies(weather, prefix="weather").astype(int)

    def target_lag(lag):
        shift = lag - horizon
        if shift < 0:
            raise ValueError(f"target lag {lag} is not known for horizon {horizon}")
        return d["BikeCount"].shift(shift)

    lag_cols = []
    for lag in [1, 2, 3, 24, 168]:
        col = f"bike_count_target_minus_{lag}"
        d[col] = target_lag(lag)
        lag_cols.append(col)

    d["rolling_24h_mean"] = target_lag(horizon).rolling(24).mean()
    d["rolling_3h_mean"] = target_lag(horizon).rolling(3).mean()
    rolling_cols = ["rolling_3h_mean", "rolling_24h_mean"]

    feature_cols = [
        "target_hour", "target_weekday", "target_month", "target_is_weekend",
        "target_hour_sin", "target_hour_cos", "target_month_sin", "target_month_cos",
        "target_temperature", "target_humidity", "target_rain", "target_wind",
        *lag_cols, *rolling_cols,
    ]

    supervised = pd.concat([d[feature_cols + ["target"]], weather_dummies], axis=1)
    supervised = supervised.dropna().reset_index(drop=True)
    return supervised.drop(columns=["target"]), supervised["target"]


In [ ]:
if TEST_PATH is None:
    raise ValueError("Set TEST_PATH to the provided hidden test .xlsx file before running all cells.")

model_path = Path(MODEL_PATH)
if not model_path.exists():
    model_path = Path("submission") / MODEL_PATH

bundle = joblib.load(model_path)
model = bundle["model"]
feature_columns = bundle["feature_columns"]

test_df = load_clean(TEST_PATH)
X_test, y_test = make_supervised_frame(test_df)
X_test = X_test.reindex(columns=feature_columns, fill_value=0)

predictions = np.clip(model.predict(X_test), 0, None)
mse = mean_squared_error(y_test, predictions)


In [ ]:
print(f"MSE: {mse:.2f}")